<a href="https://colab.research.google.com/github/blood097/Scientific_materials/blob/main/YOLO_nodule_infection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opencv-python
!pip install ultralytics
!pip install onnx

!yolo settings sync=False

In [6]:
import cv2
import numpy as np
from ultralytics import YOLO
import csv
from pathlib import Path

# Загрузка модели и изображений
model = YOLO('/content/Inf_cell_v26L-seg.pt')
input_dir = Path("/content")  # Папка с изображениями
output_csv = Path("results.csv")  # Файл, куда запишется результат

# Расширения файлов изображений
image_extensions = ("*.jpg", "*.jpeg", "*.png")

# Создание таблицы результатов
with open(output_csv, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["File Name",
                    "Total Pixels",
                    "Mask Pixels",
                    "Coverage Percentage",
                    "Objects Detected"])

# Поиск всех изображений для анализа
image_files = []
for ext in image_extensions:
    image_files.extend(input_dir.glob(ext))
print(f"Найдено изображений для обработки: {len(image_files)}")

# Основной цикл обработки
for img_path in image_files:
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Ошибка загрузки файла (пропущен): {img_path.name}")
            continue

        h, w, _ = img.shape
        total_pixels = h * w

        # Запуск сегментации для конкретного кадра
        results = model(img_path, verbose=False)

        # Создание пустой маски
        all_masks = np.zeros((h, w), dtype=np.uint8)
        object_count = 0

        for result in results:
            if result.masks is not None:
                object_count += len(result.masks.xy)
                # Отрисовка полигонов на маске
                for segments in result.masks.xy:
                    points = np.array(segments, dtype=np.int32)
                    cv2.fillPoly(all_masks, [points], 255)

        # Учет количества закрашенных пикселей маски
        mask_pixels = int(np.sum(all_masks == 255))

        # Вычисление покрытия
        coverage_pct = (
            (mask_pixels / total_pixels) * 100 if total_pixels > 0 else 0
        )

        # Заполнение результатов текущей обработки изображения в CSV
        with open(output_csv, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(
                [
                    img_path.name,
                    total_pixels,
                    mask_pixels,
                    round(coverage_pct, 4),
                    object_count,
                ]
            )

        print(f"✅ Обработан {img_path.name} | Покрытие: {coverage_pct:.2f}%")

    except Exception as e:
        print(f"Ошибка при обработке {img_path.name}: {e}")

print(f"Обработка завершена. Результаты сохранены в {output_csv.resolve()}")

Найдено изображений для обработки: 1
✅ Обработан: Test2.jpg | Покрытие: 67.64%
🎉 Обработка завершена! Результаты сохранены в: /content/results.csv
